# English -> Urdu NMT: Transformer vs. LSTM vs. Fine-tuned MarianMT
Self-contained training pipeline for Assignment 3, Question 1 (UMC005 corpus).

**Before running:** in the notebook's right-hand Settings panel, set
**Accelerator = GPU (T4 x2 or P100)** and **Internet = On**, then Run All.

At the end, `outputs.zip` will appear under the notebook's Output/Data tab —
download it and unzip `artifacts/` and `results/` into your local project
folder (next to `src/`) to use the trained checkpoints in the Streamlit GUI
and to pull real numbers into the report.

In [ ]:
!pip install -q tokenizers sacrebleu rouge-score sentencepiece

In [ ]:
import os
os.makedirs('src', exist_ok=True)


In [ ]:
%%writefile src/utils.py
"""Shared constants and small helpers used across the training/eval scripts."""
import random
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
PROCESSED_DIR = ROOT / "data" / "processed"
TOKENIZER_DIR = ROOT / "artifacts" / "tokenizers"
CHECKPOINT_DIR = ROOT / "artifacts" / "checkpoints"
RESULTS_DIR = ROOT / "results"

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
SPECIAL_TOKENS = [PAD, SOS, EOS, UNK]
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

VOCAB_SIZE = 8000
MAX_LEN = 80


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_device():
    import torch

    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            return torch.device("cuda")
        except RuntimeError:
            return torch.device("cpu")
    return torch.device("cpu")


def read_lines(path: Path) -> list[str]:
    return path.read_text(encoding="utf-8").splitlines()


def shape_urdu(text: str) -> str:
    """Reshape + bidi-reorder Urdu (Arabic-script) text for renderers that
    don't perform contextual letter-shaping themselves (matplotlib, PIL).
    Browsers and LaTeX with proper RTL support don't need this.
    """
    import arabic_reshaper
    from bidi.algorithm import get_display

    return get_display(arabic_reshaper.reshape(text))


class WhitespaceTokenizer:
    """Simple whitespace tokenizer for `rouge_score.RougeScorer`.

    `rouge_score`'s default tokenizer matches only `[a-z0-9]`, which silently
    strips all Urdu (Arabic-script) text to an empty token list and forces
    every ROUGE score to 0 regardless of translation quality. Our corpus and
    tokenizer output are already whitespace-segmented (words and punctuation
    separated by spaces), so a plain `.split()` is a correct, script-agnostic
    substitute.
    """

    def tokenize(self, text: str) -> list[str]:
        return text.split()


In [ ]:
%%writefile src/data_prep.py
"""Download, clean, and split the UMC005 English-Urdu parallel corpus.

Combines the freely-redistributable Quran and Bible sections of UMC005 (the
Penn/Emille sections are excluded by the corpus authors for licensing
reasons) into a single train/dev/test split under data/processed/.
"""
import io
import re
import random
import zipfile
from pathlib import Path
from urllib.request import urlopen

CORPUS_URL = "https://ufal.mff.cuni.cz/umc/005-en-ur/download.php?f=umc005-corpus.zip"
ROOT = Path(__file__).resolve().parent.parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
SECTIONS = ["quran", "bible"]
SPLITS = ["train", "dev", "test"]

BOM = "﻿"
WHITESPACE_RE = re.compile(r"\s+")


def download_corpus() -> None:
    """Fetch and extract the UMC005 zip if not already present locally."""
    zip_path = RAW_DIR / "umc005-corpus.zip"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if not zip_path.exists():
        with urlopen(CORPUS_URL) as resp:
            zip_path.write_bytes(resp.read())
    if not (RAW_DIR / "quran").exists():
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(RAW_DIR)


def clean_line(line: str) -> str:
    line = line.replace(BOM, "").strip()
    line = WHITESPACE_RE.sub(" ", line)
    return line


def load_pairs(section: str, split: str) -> list[tuple[str, str]]:
    en_path = RAW_DIR / section / f"{split}.en"
    ur_path = RAW_DIR / section / f"{split}.ur"
    en_lines = en_path.read_text(encoding="utf-8").splitlines()
    ur_lines = ur_path.read_text(encoding="utf-8").splitlines()
    assert len(en_lines) == len(ur_lines), f"{section}/{split}: misaligned line counts"
    return list(zip(en_lines, ur_lines))


def clean_and_filter(pairs: list[tuple[str, str]], min_len=1, max_len=100) -> list[tuple[str, str]]:
    seen = set()
    out = []
    for en, ur in pairs:
        en, ur = clean_line(en), clean_line(ur)
        if not en or not ur:
            continue
        en_wc, ur_wc = len(en.split()), len(ur.split())
        if not (min_len <= en_wc <= max_len and min_len <= ur_wc <= max_len):
            continue
        key = (en, ur)
        if key in seen:
            continue
        seen.add(key)
        out.append((en, ur))
    return out


def write_split(pairs: list[tuple[str, str]], split: str) -> None:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / f"{split}.en").write_text(
        "\n".join(en for en, _ in pairs) + "\n", encoding="utf-8"
    )
    (PROCESSED_DIR / f"{split}.ur").write_text(
        "\n".join(ur for _, ur in pairs) + "\n", encoding="utf-8"
    )


def main(seed: int = 42) -> None:
    download_corpus()
    random.seed(seed)

    for split in SPLITS:
        pairs: list[tuple[str, str]] = []
        for section in SECTIONS:
            pairs.extend(load_pairs(section, split))
        pairs = clean_and_filter(pairs)
        random.shuffle(pairs)
        write_split(pairs, split)
        print(f"{split}: {len(pairs)} sentence pairs")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/tokenizer_train.py
"""Train independent Byte-Level BPE tokenizers for English and Urdu.

Each language gets its own tokenizer (different scripts/morphology), trained
on the training split only, and saved as a single-file HF `tokenizers` JSON
that `dataset.py` and the model scripts load at run time.
"""
from tokenizers import ByteLevelBPETokenizer

from utils import PROCESSED_DIR, SPECIAL_TOKENS, TOKENIZER_DIR, VOCAB_SIZE


def train_tokenizer(lang: str) -> None:
    train_file = str(PROCESSED_DIR / f"train.{lang}")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train(
        files=[train_file],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
    )
    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(TOKENIZER_DIR / f"{lang}.json"))
    print(f"{lang}: vocab size = {tokenizer.get_vocab_size()}")


def main() -> None:
    for lang in ("en", "ur"):
        train_tokenizer(lang)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/dataset.py
"""PyTorch Dataset + collate function for the English-Urdu parallel corpus."""
from pathlib import Path

import torch
from tokenizers import Tokenizer
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

from utils import EOS_ID, MAX_LEN, PAD_ID, PROCESSED_DIR, SOS_ID, TOKENIZER_DIR, read_lines


def load_tokenizers() -> tuple[Tokenizer, Tokenizer]:
    en_tok = Tokenizer.from_file(str(TOKENIZER_DIR / "en.json"))
    ur_tok = Tokenizer.from_file(str(TOKENIZER_DIR / "ur.json"))
    return en_tok, ur_tok


class TranslationDataset(Dataset):
    """Tokenizes an (en, ur) split on the fly and wraps each side with <sos>/<eos>."""

    def __init__(self, split: str, en_tok: Tokenizer, ur_tok: Tokenizer, max_len: int = MAX_LEN):
        self.en_lines = read_lines(PROCESSED_DIR / f"{split}.en")
        self.ur_lines = read_lines(PROCESSED_DIR / f"{split}.ur")
        assert len(self.en_lines) == len(self.ur_lines)
        self.en_tok, self.ur_tok = en_tok, ur_tok
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.en_lines)

    def _encode(self, tok: Tokenizer, text: str) -> list[int]:
        ids = tok.encode(text).ids[: self.max_len - 2]
        return [SOS_ID] + ids + [EOS_ID]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        src = self._encode(self.en_tok, self.en_lines[idx])
        tgt = self._encode(self.ur_tok, self.ur_lines[idx])
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


def collate_fn(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    src_batch, tgt_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)
    return {"src": src_padded, "tgt": tgt_padded}


In [ ]:
%%writefile src/transformer_model.py
"""A from-scratch implementation of the Transformer encoder-decoder
(Vaswani et al., 2017) for sequence-to-sequence translation.

Everything here — scaled dot-product attention, multi-head attention,
sinusoidal positional encoding, the position-wise feed-forward block, and
the encoder/decoder stacks — is implemented directly with tensor ops rather
than `nn.Transformer`, per the assignment requirement to build the
architecture from scratch.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from utils import PAD_ID


class PositionalEncoding(nn.Module):
    """Fixed sinusoidal position embeddings, added to token embeddings."""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class MultiHeadAttention(nn.Module):
    """Scaled dot-product attention, computed in parallel across `n_heads`."""

    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        b = query.size(0)
        q = self.w_q(query).view(b, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(b, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(b, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(b, -1, self.n_heads * self.d_k)
        return self.w_o(out), attn


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        attn_out, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))
        cross_out, cross_attn = self.cross_attn(x, memory, memory, src_mask)
        x = self.norm2(x + self.dropout(cross_out))
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x, cross_attn


class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 256,
        n_heads: int = 8,
        n_layers: int = 4,
        d_ff: int = 1024,
        dropout: float = 0.1,
        max_len: int = 80,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_ID)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    @staticmethod
    def make_src_mask(src: torch.Tensor) -> torch.Tensor:
        return (src != PAD_ID).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)

    @staticmethod
    def make_tgt_mask(tgt: torch.Tensor) -> torch.Tensor:
        pad_mask = (tgt != PAD_ID).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)
        seq_len = tgt.size(1)
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=tgt.device)).bool()
        return pad_mask & causal_mask  # (B, 1, T, T)

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        x = self.dropout(self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model)))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, memory, src_mask, tgt_mask):
        x = self.dropout(self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model)))
        cross_attn = None
        for layer in self.decoder_layers:
            x, cross_attn = layer(x, memory, src_mask, tgt_mask)
        return x, cross_attn

    def forward(self, src: torch.Tensor, tgt: torch.Tensor):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        memory = self.encode(src, src_mask)
        dec_out, cross_attn = self.decode(tgt, memory, src_mask, tgt_mask)
        logits = self.generator(dec_out)
        return logits, cross_attn

    @torch.no_grad()
    def greedy_decode(self, src: torch.Tensor, sos_id: int, eos_id: int, max_len: int = 80):
        """Autoregressive greedy decoding, one sentence (batch size 1) at a time.
        Returns the generated token ids and the final step's cross-attention weights.
        """
        self.eval()
        src_mask = self.make_src_mask(src)
        memory = self.encode(src, src_mask)
        ys = torch.full((1, 1), sos_id, dtype=torch.long, device=src.device)
        cross_attn = None
        for _ in range(max_len - 1):
            tgt_mask = self.make_tgt_mask(ys)
            dec_out, cross_attn = self.decode(ys, memory, src_mask, tgt_mask)
            logits = self.generator(dec_out[:, -1])
            next_token = logits.argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_token], dim=1)
            if next_token.item() == eos_id:
                break
        return ys.squeeze(0), cross_attn


In [ ]:
%%writefile src/lstm_model.py
"""Seq2seq LSTM baseline with Luong (multiplicative) attention.

Architectural counterpart to the from-scratch Transformer: a bidirectional
LSTM encoder, an autoregressive LSTM decoder, and a Luong-style attention
mechanism over the encoder outputs at every decoding step. Used purely as a
comparison point in the report (training time, BLEU/ROUGE, perplexity, etc.)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils import PAD_ID


class Encoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.bridge_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.bridge_c = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor):
        lengths = (src != PAD_ID).sum(dim=1).clamp(min=1).cpu()
        embedded = self.dropout(self.embed(src))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, (h, c) = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=src.size(1)
        )
        # h, c: (2, B, hidden) from the single bidirectional layer -> merge into decoder's initial state
        h_cat = torch.cat([h[0], h[1]], dim=-1)
        c_cat = torch.cat([c[0], c[1]], dim=-1)
        h0 = torch.tanh(self.bridge_h(h_cat)).unsqueeze(0)
        c0 = torch.tanh(self.bridge_c(c_cat)).unsqueeze(0)
        return outputs, (h0, c0)


class LuongAttention(nn.Module):
    """General (bilinear) attention: score(h_t, h_s) = h_t^T W h_s."""

    def __init__(self, hidden_dim: int, encoder_dim: int):
        super().__init__()
        self.W = nn.Linear(encoder_dim, hidden_dim, bias=False)

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, src_mask: torch.Tensor):
        # dec_hidden: (B, hidden), enc_outputs: (B, S, encoder_dim)
        proj = self.W(enc_outputs)  # (B, S, hidden)
        scores = torch.bmm(proj, dec_hidden.unsqueeze(2)).squeeze(2)  # (B, S)
        scores = scores.masked_fill(~src_mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)  # (B, S)
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1)  # (B, encoder_dim)
        return context, weights


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden_dim: int, encoder_dim: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.attention = LuongAttention(hidden_dim, encoder_dim)
        self.out = nn.Linear(hidden_dim + encoder_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, input_token, hidden, enc_outputs, src_mask):
        embedded = self.dropout(self.embed(input_token))  # (B, 1, emb)
        dec_out, hidden = self.lstm(embedded, hidden)  # dec_out: (B, 1, hidden)
        context, attn_weights = self.attention(dec_out.squeeze(1), enc_outputs, src_mask)
        logits = self.out(torch.cat([dec_out.squeeze(1), context], dim=-1))
        return logits, hidden, attn_weights


class Seq2SeqLSTM(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        emb_dim: int = 256,
        hidden_dim: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hidden_dim, dropout)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hidden_dim, hidden_dim * 2, dropout)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, teacher_forcing_ratio: float = 1.0):
        src_mask = src != PAD_ID
        enc_outputs, hidden = self.encoder(src)

        batch_size, tgt_len = tgt.size()
        vocab_size = self.decoder.out.out_features
        logits_all = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        input_token = tgt[:, :1]  # <sos>
        for t in range(tgt_len - 1):
            logits, hidden, _ = self.decoder.forward_step(input_token, hidden, enc_outputs, src_mask)
            logits_all[:, t] = logits
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            input_token = tgt[:, t + 1 : t + 2] if use_teacher else logits.argmax(-1, keepdim=True)
        return logits_all, None

    @torch.no_grad()
    def greedy_decode(self, src: torch.Tensor, sos_id: int, eos_id: int, max_len: int = 80):
        self.eval()
        src_mask = src != PAD_ID
        enc_outputs, hidden = self.encoder(src)
        input_token = torch.full((1, 1), sos_id, dtype=torch.long, device=src.device)
        tokens = [sos_id]
        attn_history = []
        for _ in range(max_len - 1):
            logits, hidden, attn_weights = self.decoder.forward_step(input_token, hidden, enc_outputs, src_mask)
            next_token = logits.argmax(-1, keepdim=True)
            tokens.append(next_token.item())
            attn_history.append(attn_weights.squeeze(0))
            if next_token.item() == eos_id:
                break
            input_token = next_token
        return torch.tensor(tokens, device=src.device), torch.stack(attn_history)


In [ ]:
%%writefile src/train.py
"""Shared training loop for both the from-scratch Transformer and the LSTM
baseline: teacher forcing, LR scheduling, early stopping, gradient clipping,
checkpointing, and per-epoch CSV logging (used later for the loss-curve plot).
"""
import argparse
import csv
import json
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from dataset import TranslationDataset, collate_fn, load_tokenizers
from lstm_model import Seq2SeqLSTM
from transformer_model import Transformer
from utils import CHECKPOINT_DIR, MAX_LEN, PAD_ID, RESULTS_DIR, get_device, set_seed


class NoamScheduler:
    """Warmup + inverse-sqrt-decay LR schedule from 'Attention Is All You Need' (sec 5.3)."""

    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.d_model**-0.5 * min(self.step_num**-0.5, self.step_num * self.warmup_steps**-1.5)
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        self.optimizer.step()


def run_epoch(model, loader, optimizer, criterion, device, scheduler=None, train=True):
    model.train(train)
    total_loss, total_tokens = 0.0, 0
    for batch in loader:
        src, tgt = batch["src"].to(device), batch["tgt"].to(device)
        with torch.set_grad_enabled(train):
            if isinstance(model, Transformer):
                logits, _ = model(src, tgt[:, :-1])
            else:
                logits, _ = model(src, tgt, teacher_forcing_ratio=1.0 if train else 1.0)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if scheduler is not None:
                scheduler.step()
            else:
                optimizer.step()
        n_tokens = (tgt[:, 1:] != PAD_ID).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
    return total_loss / max(total_tokens, 1)


def train_model(model_name: str, epochs: int, batch_size: int, patience: int = 5, lr: float = 3e-4):
    set_seed(42)
    device = get_device()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    print(f"Training {model_name} on device={device}")

    en_tok, ur_tok = load_tokenizers()
    train_ds = TranslationDataset("train", en_tok, ur_tok)
    dev_ds = TranslationDataset("dev", en_tok, ur_tok)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    src_vocab_size = en_tok.get_vocab_size()
    tgt_vocab_size = ur_tok.get_vocab_size()

    if model_name == "transformer":
        model = Transformer(src_vocab_size, tgt_vocab_size, max_len=MAX_LEN).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
        scheduler = NoamScheduler(optimizer, d_model=model.d_model, warmup_steps=2000)
        plateau_scheduler = None
    elif model_name == "lstm":
        model = Seq2SeqLSTM(src_vocab_size, tgt_vocab_size).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = None
        plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    else:
        raise ValueError(model_name)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"{model_name}: {n_params:,} parameters")

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    log_path = RESULTS_DIR / f"{model_name}_log.csv"
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    start_time = time.time()
    peak_mem_mb = 0.0

    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "lr", "elapsed_sec"])

        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            train_loss = run_epoch(model, train_loader, optimizer, criterion, device, scheduler, train=True)
            val_loss = run_epoch(model, dev_loader, optimizer, criterion, device, scheduler=None, train=False)

            if plateau_scheduler is not None:
                plateau_scheduler.step(val_loss)
            current_lr = optimizer.param_groups[0]["lr"]

            if device.type == "cuda":
                peak_mem_mb = max(peak_mem_mb, torch.cuda.max_memory_allocated() / 1e6)

            elapsed = time.time() - epoch_start
            writer.writerow([epoch, f"{train_loss:.4f}", f"{val_loss:.4f}", f"{current_lr:.6f}", f"{elapsed:.1f}"])
            f.flush()
            print(
                f"[{model_name}] epoch {epoch}/{epochs} train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} lr={current_lr:.6f} ({elapsed:.1f}s)"
            )

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                epochs_without_improvement = 0
                torch.save(model.state_dict(), CHECKPOINT_DIR / f"{model_name}_best.pt")
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    print(f"[{model_name}] early stopping at epoch {epoch} (patience={patience})")
                    break

    total_time = time.time() - start_time
    meta = {
        "model": model_name,
        "n_params": n_params,
        "best_val_loss": best_val_loss,
        "training_time_sec": total_time,
        "peak_gpu_mem_mb": peak_mem_mb,
        "device": str(device),
        "epochs_run": epoch,
    }
    with open(RESULTS_DIR / f"{model_name}_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"[{model_name}] done in {total_time:.1f}s, best_val_loss={best_val_loss:.4f}")
    return meta


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=["transformer", "lstm"], required=True)
    parser.add_argument("--epochs", type=int, default=25)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--patience", type=int, default=5)
    args = parser.parse_args()
    train_model(args.model, args.epochs, args.batch_size, args.patience)


In [ ]:
%%writefile src/evaluate.py
"""Evaluate a trained model on the test set: BLEU, ROUGE-1/2/L, perplexity,
inference speed, and (for the two from-scratch models) parameter count /
training time / memory pulled from the training-run metadata JSON.
"""
import argparse
import json
import math
import time

import psutil
import sacrebleu
import torch
import torch.nn as nn
from rouge_score import rouge_scorer
from torch.utils.data import DataLoader

from dataset import TranslationDataset, collate_fn, load_tokenizers
from lstm_model import Seq2SeqLSTM
from train import run_epoch
from transformer_model import Transformer
from utils import CHECKPOINT_DIR, EOS_ID, MAX_LEN, PAD_ID, RESULTS_DIR, SOS_ID, WhitespaceTokenizer, get_device


def decode_greedy(model, src_tensor, sos_id, eos_id, max_len):
    ids, attn = model.greedy_decode(src_tensor, sos_id, eos_id, max_len)
    return ids, attn


def evaluate_model(model_name: str, batch_size: int = 32, max_len: int = MAX_LEN):
    device = get_device()
    en_tok, ur_tok = load_tokenizers()
    test_ds = TranslationDataset("test", en_tok, ur_tok, max_len=max_len)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    src_vocab_size, tgt_vocab_size = en_tok.get_vocab_size(), ur_tok.get_vocab_size()
    if model_name == "transformer":
        model = Transformer(src_vocab_size, tgt_vocab_size, max_len=max_len)
    elif model_name == "lstm":
        model = Seq2SeqLSTM(src_vocab_size, tgt_vocab_size)
    else:
        raise ValueError(model_name)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / f"{model_name}_best.pt", map_location=device))
    model.to(device).eval()

    # Perplexity from cross-entropy loss over the test set.
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.0)
    test_loss = run_epoch(model, test_loader, optimizer=None, criterion=criterion, device=device, train=False)
    perplexity = math.exp(min(test_loss, 20))

    # Greedy decode sentence-by-sentence for BLEU/ROUGE + inference speed.
    hyps, refs, srcs = [], [], []
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1e6
    start = time.time()
    for i in range(len(test_ds)):
        src_tensor, tgt_tensor = test_ds[i]
        src_tensor = src_tensor.unsqueeze(0).to(device)
        ids, _ = decode_greedy(model, src_tensor, SOS_ID, EOS_ID, max_len)
        hyp_ids = [t for t in ids.tolist() if t not in (SOS_ID, EOS_ID, PAD_ID)]
        ref_ids = [t for t in tgt_tensor.tolist() if t not in (SOS_ID, EOS_ID, PAD_ID)]
        hyps.append(ur_tok.decode(hyp_ids))
        refs.append(ur_tok.decode(ref_ids))
        srcs.append(en_tok.decode([t for t in src_tensor.squeeze(0).tolist() if t not in (SOS_ID, EOS_ID, PAD_ID)]))
    elapsed = time.time() - start
    mem_after = process.memory_info().rss / 1e6

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False, tokenizer=WhitespaceTokenizer()
    )
    rouge_sums = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for hyp, ref in zip(hyps, refs):
        scores = scorer.score(ref, hyp)
        for k in rouge_sums:
            rouge_sums[k] += scores[k].fmeasure
    rouge_avg = {k: v / len(hyps) for k, v in rouge_sums.items()}

    n_params = sum(p.numel() for p in model.parameters())
    meta_path = RESULTS_DIR / f"{model_name}_meta.json"
    train_meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}

    results = {
        "model": model_name,
        "bleu": bleu.score,
        "rouge1_f": rouge_avg["rouge1"],
        "rouge2_f": rouge_avg["rouge2"],
        "rougeL_f": rouge_avg["rougeL"],
        "perplexity": perplexity,
        "test_loss": test_loss,
        "n_params": n_params,
        "inference_sentences_per_sec": len(test_ds) / elapsed,
        "inference_mem_delta_mb": mem_after - mem_before,
        "training_time_sec": train_meta.get("training_time_sec"),
        "peak_gpu_mem_mb": train_meta.get("peak_gpu_mem_mb"),
        "device": str(device),
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_DIR / f"{model_name}_eval.json", "w") as f:
        json.dump(results, f, indent=2)

    samples = [{"src": s, "ref": r, "hyp": h} for s, r, h in list(zip(srcs, refs, hyps))[:20]]
    with open(RESULTS_DIR / f"{model_name}_samples.json", "w", encoding="utf-8") as f:
        json.dump(samples, f, indent=2, ensure_ascii=False)

    print(json.dumps(results, indent=2))
    return results


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=["transformer", "lstm"], required=True)
    args = parser.parse_args()
    evaluate_model(args.model)


In [ ]:
%%writefile src/attention_viz.py
"""Visualize the Transformer's decoder-to-encoder cross-attention: for a
handful of test sentences, render a heatmap of which English source tokens
the model attended to while generating each Urdu target token.
"""
import argparse

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch

matplotlib.rcParams["font.family"] = ["Tahoma", "Segoe UI", "Noto Nastaliq Urdu", "Noto Sans Arabic", "Arial", "sans-serif"]

from dataset import TranslationDataset, load_tokenizers
from transformer_model import Transformer
from utils import CHECKPOINT_DIR, EOS_ID, MAX_LEN, PAD_ID, RESULTS_DIR, SOS_ID, get_device, shape_urdu


def plot_attention(src_tokens: list[str], tgt_tokens: list[str], attn: np.ndarray, out_path):
    # Arabic-script tokens need contextual reshaping + bidi reordering; matplotlib
    # (unlike a browser or a proper LaTeX RTL setup) doesn't shape text itself.
    shaped_tgt_tokens = [shape_urdu(t) for t in tgt_tokens]

    fig, ax = plt.subplots(figsize=(max(6, len(src_tokens) * 0.6), max(4, len(tgt_tokens) * 0.5)))
    im = ax.imshow(attn, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_yticklabels(shaped_tgt_tokens, fontsize=11)
    ax.set_xlabel("English source tokens")
    ax.set_ylabel("Urdu generated tokens")
    fig.colorbar(im, ax=ax, label="attention weight")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


@torch.no_grad()
def generate_with_full_attention(model, src_tensor, sos_id, eos_id, max_len):
    """Greedy decode while recording the cross-attention row produced at
    every step, so we get a full (tgt_len x src_len) matrix rather than just
    the final step's attention.
    """
    device = src_tensor.device
    src_mask = model.make_src_mask(src_tensor)
    memory = model.encode(src_tensor, src_mask)
    ys = torch.full((1, 1), sos_id, dtype=torch.long, device=device)
    attn_rows = []
    for _ in range(max_len - 1):
        tgt_mask = model.make_tgt_mask(ys)
        dec_out, cross_attn = model.decode(ys, memory, src_mask, tgt_mask)
        # cross_attn: (B, n_heads, T, S) -> average heads, take last query position
        last_step_attn = cross_attn[0, :, -1, :].mean(dim=0)  # (S,)
        attn_rows.append(last_step_attn.cpu().numpy())
        logits = model.generator(dec_out[:, -1])
        next_token = logits.argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)
        if next_token.item() == eos_id:
            break
    return ys.squeeze(0), np.stack(attn_rows)


def main(n_examples: int = 6):
    device = get_device()
    en_tok, ur_tok = load_tokenizers()
    test_ds = TranslationDataset("test", en_tok, ur_tok)

    model = Transformer(en_tok.get_vocab_size(), ur_tok.get_vocab_size(), max_len=MAX_LEN)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / "transformer_best.pt", map_location=device))
    model.to(device).eval()

    out_dir = RESULTS_DIR / "attention_examples"
    out_dir.mkdir(parents=True, exist_ok=True)

    for i in range(min(n_examples, len(test_ds))):
        src_tensor, _ = test_ds[i]
        src_ids = [t for t in src_tensor.tolist() if t not in (SOS_ID, EOS_ID, PAD_ID)]
        src_tokens = [en_tok.decode([tid]).strip() or "?" for tid in src_ids]

        src_input = src_tensor.unsqueeze(0).to(device)
        gen_ids, attn_matrix = generate_with_full_attention(model, src_input, SOS_ID, EOS_ID, MAX_LEN)
        gen_ids_list = [t for t in gen_ids.tolist() if t not in (SOS_ID, EOS_ID, PAD_ID)]
        tgt_tokens = [ur_tok.decode([tid]).strip() or "?" for tid in gen_ids_list]

        # attn_matrix rows = one per generated step; columns = full src_tensor incl. <sos>/<eos>.
        # Keep only real (non-EOS) target rows and drop the <sos>/<eos> source columns.
        attn_matrix = attn_matrix[: len(tgt_tokens), 1 : 1 + len(src_tokens)]

        plot_attention(src_tokens, tgt_tokens, attn_matrix, out_dir / f"example_{i}.png")
        print(f"saved {out_dir / f'example_{i}.png'}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n", type=int, default=6)
    args = parser.parse_args()
    main(args.n)


In [ ]:
%%writefile src/finetune_pretrained.py
"""Bonus task: fine-tune the pretrained Helsinki-NLP/opus-mt-en-ur MarianMT
model on the same UMC005 train split, then evaluate it with the same
BLEU/ROUGE/perplexity metrics as the from-scratch models for a direct
custom-vs-pretrained comparison.
"""
import json
import math
import time

import psutil
import sacrebleu
import torch
from datasets import Dataset
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from utils import CHECKPOINT_DIR, PROCESSED_DIR, RESULTS_DIR, WhitespaceTokenizer, get_device, read_lines

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ur"
OUT_DIR = CHECKPOINT_DIR / "finetuned_opus_mt"


def load_split(split: str) -> Dataset:
    en = read_lines(PROCESSED_DIR / f"{split}.en")
    ur = read_lines(PROCESSED_DIR / f"{split}.ur")
    return Dataset.from_dict({"en": en, "ur": ur})


def main(epochs: int = 3, batch_size: int = 16, max_len: int = 80):
    device = get_device()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

    train_ds, dev_ds, test_ds = load_split("train"), load_split("dev"), load_split("test")

    def preprocess(batch):
        model_inputs = tokenizer(batch["en"], max_length=max_len, truncation=True)
        labels = tokenizer(text_target=batch["ur"], max_length=max_len, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=["en", "ur"])
    dev_tok = dev_ds.map(preprocess, batched=True, remove_columns=["en", "ur"])

    collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    args = Seq2SeqTrainingArguments(
        output_dir=str(OUT_DIR),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        predict_with_generate=False,
        logging_steps=50,
        report_to=[],
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=collator,
        processing_class=tokenizer,
    )

    start = time.time()
    trainer.train()
    training_time = time.time() - start
    trainer.save_model(str(OUT_DIR))
    tokenizer.save_pretrained(str(OUT_DIR))

    # --- Evaluation on the held-out test split ---
    model.eval()
    hyps, refs = [], []
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1e6
    infer_start = time.time()
    with torch.no_grad():
        for i in range(0, len(test_ds), batch_size):
            batch = test_ds[i : i + batch_size]
            inputs = tokenizer(batch["en"], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(device)
            generated = model.generate(**inputs, max_length=max_len)
            hyps.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
            refs.extend(batch["ur"])
    infer_elapsed = time.time() - infer_start
    mem_after = process.memory_info().rss / 1e6

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False, tokenizer=WhitespaceTokenizer()
    )
    rouge_sums = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for hyp, ref in zip(hyps, refs):
        scores = scorer.score(ref, hyp)
        for k in rouge_sums:
            rouge_sums[k] += scores[k].fmeasure
    rouge_avg = {k: v / len(hyps) for k, v in rouge_sums.items()}

    eval_loss = trainer.evaluate()["eval_loss"]
    perplexity = math.exp(min(eval_loss, 20))
    n_params = sum(p.numel() for p in model.parameters())

    results = {
        "model": "finetuned_opus_mt",
        "bleu": bleu.score,
        "rouge1_f": rouge_avg["rouge1"],
        "rouge2_f": rouge_avg["rouge2"],
        "rougeL_f": rouge_avg["rougeL"],
        "perplexity": perplexity,
        "test_loss": eval_loss,
        "n_params": n_params,
        "inference_sentences_per_sec": len(test_ds) / infer_elapsed,
        "inference_mem_delta_mb": mem_after - mem_before,
        "training_time_sec": training_time,
        "device": str(device),
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_DIR / "finetuned_opus_mt_eval.json", "w") as f:
        json.dump(results, f, indent=2)

    samples = [{"src": s, "ref": r, "hyp": h} for s, r, h in list(zip(test_ds["en"], refs, hyps))[:20]]
    with open(RESULTS_DIR / "finetuned_opus_mt_samples.json", "w", encoding="utf-8") as f:
        json.dump(samples, f, indent=2, ensure_ascii=False)

    print(json.dumps(results, indent=2))


if __name__ == "__main__":
    main()


## 1. Data preparation

In [ ]:
import sys
sys.path.insert(0, 'src')
import data_prep
data_prep.main()

## 2. Train BPE tokenizers

In [ ]:
import tokenizer_train
tokenizer_train.main()

## 3. Train the from-scratch Transformer
GPU makes larger batches cheap, so we bump batch size vs. the CPU config.

In [ ]:
import importlib
import train
importlib.reload(train)
transformer_meta = train.train_model('transformer', epochs=30, batch_size=128, patience=5)

## 4. Train the LSTM baseline

In [ ]:
lstm_meta = train.train_model('lstm', epochs=30, batch_size=128, patience=5)

## 5. Evaluate both models (BLEU / ROUGE / perplexity / speed)

In [ ]:
import evaluate
transformer_results = evaluate.evaluate_model('transformer')
lstm_results = evaluate.evaluate_model('lstm')

## 6. Attention visualization

In [ ]:
import attention_viz
attention_viz.main(n_examples=6)

## 7. Loss curve plot (train vs. val, both models)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for name, color in [('transformer', 'tab:blue'), ('lstm', 'tab:orange')]:
    df = pd.read_csv(f'results/{name}_log.csv')
    ax.plot(df['epoch'], df['train_loss'], color=color, linestyle='-', label=f'{name} train')
    ax.plot(df['epoch'], df['val_loss'], color=color, linestyle='--', label=f'{name} val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.set_title('Training / Validation Loss')
fig.tight_layout()
fig.savefig('results/loss_curves.png', dpi=150)
plt.show()

## 8. Bonus: fine-tune the pretrained MarianMT model
Fine-tunes `Helsinki-NLP/opus-mt-en-ur` on the same train split for comparison.

In [ ]:
!pip install -q transformers datasets accelerate
import finetune_pretrained
finetune_pretrained.main()

## 9. Package everything for download
We deliberately **exclude** the fine-tuned MarianMT checkpoint's raw weights here --
with optimizer state it can be 500MB-1GB and isn't needed locally (the GUI only
serves the two from-scratch models; the bonus model's *metrics* are all the report
needs, and those live in `results/`). This keeps the zip small and the download reliable.

In [ ]:
import zipfile
from pathlib import Path

INCLUDE = [
    'artifacts/checkpoints/transformer_best.pt',
    'artifacts/checkpoints/lstm_best.pt',
]
with zipfile.ZipFile('/kaggle/working/outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in INCLUDE:
        if Path(f).exists():
            z.write(f, arcname=f)
    for pattern_dir in ['artifacts/tokenizers', 'results']:
        for p in Path(pattern_dir).rglob('*'):
            if p.is_file():
                z.write(p, arcname=p)

size_mb = Path('/kaggle/working/outputs.zip').stat().st_size / 1e6
print(f'Wrote /kaggle/working/outputs.zip ({size_mb:.1f} MB).')
print('If the Output pane download does not work, use Save Version -> Save & Run All (Commit),\n'
      'then open the committed version and download outputs.zip from its Output tab.')